# Isaac Sim Single Scene Collection

This notebook provides **20 single-scene Isaac Sim scripts**.

Each scene is designed to be copied into the **Isaac Sim Script Editor** and run independently.
All code is written in **English only** and follows a **native Isaac Sim asset loading style** based on:

- `get_assets_root_path()`
- `add_reference_to_stage()`

## Usage

1. Open Isaac Sim.
2. Open **Window → Script Editor**.
3. Run the **Utility Cell** first.
4. Run **one scene cell at a time**.

Each scene creates a single environment under `/World/<SceneName>`.


## Utility Cell


In [ ]:
import carb
import omni.usd
import numpy as np

from pxr import Gf, Sdf, UsdGeom, UsdLux, UsdPhysics

from isaacsim.core.utils.stage import add_reference_to_stage
from isaacsim.storage.native import get_assets_root_path


ASSETS_ROOT_PATH = get_assets_root_path()
if ASSETS_ROOT_PATH is None:
    raise RuntimeError("Could not find Isaac Sim assets root path.")


def get_stage():
    return omni.usd.get_context().get_stage()


def ensure_world():
    stage = get_stage()
    if not stage.GetPrimAtPath("/World").IsValid():
        UsdGeom.Xform.Define(stage, "/World")


def clear_prim(prim_path: str):
    stage = get_stage()
    prim = stage.GetPrimAtPath(prim_path)
    if prim.IsValid():
        stage.RemovePrim(prim_path)


def create_xform(prim_path: str):
    stage = get_stage()
    return UsdGeom.Xform.Define(stage, prim_path)


def set_translate(prim_path: str, translation):
    stage = get_stage()
    prim = stage.GetPrimAtPath(prim_path)
    xformable = UsdGeom.Xformable(prim)
    translate_ops = [op for op in xformable.GetOrderedXformOps() if op.GetOpType() == UsdGeom.XformOp.TypeTranslate]
    if translate_ops:
        translate_ops[0].Set(Gf.Vec3d(*translation))
    else:
        xformable.AddTranslateOp().Set(Gf.Vec3d(*translation))


def set_scale(prim_path: str, scale):
    stage = get_stage()
    prim = stage.GetPrimAtPath(prim_path)
    xformable = UsdGeom.Xformable(prim)
    scale_ops = [op for op in xformable.GetOrderedXformOps() if op.GetOpType() == UsdGeom.XformOp.TypeScale]
    if scale_ops:
        scale_ops[0].Set(Gf.Vec3f(*scale))
    else:
        xformable.AddScaleOp().Set(Gf.Vec3f(*scale))


def set_rotate_xyz(prim_path: str, rotation_xyz_deg):
    stage = get_stage()
    prim = stage.GetPrimAtPath(prim_path)
    xformable = UsdGeom.Xformable(prim)
    rotate_ops = [op for op in xformable.GetOrderedXformOps() if op.GetOpType() == UsdGeom.XformOp.TypeRotateXYZ]
    if rotate_ops:
        rotate_ops[0].Set(Gf.Vec3f(*rotation_xyz_deg))
    else:
        xformable.AddRotateXYZOp().Set(Gf.Vec3f(*rotation_xyz_deg))


def add_light(light_path: str = "/World/DistantLight", intensity: float = 3000.0):
    stage = get_stage()
    if stage.GetPrimAtPath(light_path).IsValid():
        return
    light = UsdLux.DistantLight.Define(stage, Sdf.Path(light_path))
    light.CreateIntensityAttr(float(intensity))


def add_ground_plane(prim_path: str = "/World/GroundPlane", size: float = 100.0):
    stage = get_stage()
    if stage.GetPrimAtPath(prim_path).IsValid():
        return
    plane = UsdGeom.Mesh.Define(stage, prim_path)
    half = size * 0.5
    points = [
        (-half, -half, 0.0),
        ( half, -half, 0.0),
        ( half,  half, 0.0),
        (-half,  half, 0.0),
    ]
    plane.CreatePointsAttr(points)
    plane.CreateFaceVertexCountsAttr([4])
    plane.CreateFaceVertexIndicesAttr([0, 1, 2, 3])


def add_box(prim_path: str, translation, scale=(1.0, 1.0, 1.0)):
    stage = get_stage()
    cube = UsdGeom.Cube.Define(stage, prim_path)
    cube.CreateSizeAttr(1.0)
    set_translate(prim_path, translation)
    set_scale(prim_path, scale)
    return prim_path


def add_ramp(prim_path: str, translation, scale=(2.5, 1.2, 0.2), rotation_xyz_deg=(0.0, 0.0, 0.0)):
    add_box(prim_path, translation, scale=scale)
    set_rotate_xyz(prim_path, rotation_xyz_deg)
    return prim_path


def resolve_asset(relative_candidates):
    for relative_path in relative_candidates:
        usd_path = ASSETS_ROOT_PATH + relative_path
        if omni.client.stat(usd_path).ok:
            return usd_path
    return None


def add_asset_from_candidates(prim_path: str, relative_candidates, translation=None, rotation_xyz_deg=None, scale=None):
    usd_path = resolve_asset(relative_candidates)
    if usd_path is None:
        carb.log_warn(f"Could not resolve asset for {prim_path}. Candidates: {relative_candidates}")
        return None
    add_reference_to_stage(usd_path=usd_path, prim_path=prim_path)
    if translation is not None:
        set_translate(prim_path, translation)
    if rotation_xyz_deg is not None:
        set_rotate_xyz(prim_path, rotation_xyz_deg)
    if scale is not None:
        set_scale(prim_path, scale)
    return usd_path


def build_scene_root(scene_name: str):
    ensure_world()
    add_light()
    add_ground_plane()
    scene_root = f"/World/{scene_name}"
    clear_prim(scene_root)
    create_xform(scene_root)
    return scene_root


def add_robot(prim_path: str, robot_key: str, translation, rotation_xyz_deg=(0.0, 0.0, 0.0), scale=None):
    robot_assets = {
        "franka": [
            "/Isaac/Robots/FrankaRobotics/FrankaPanda/franka.usd",
        ],
        "nova_carter": [
            "/Isaac/Robots/Carter/nova_carter.usd",
            "/Isaac/Robots/Carter/carter_v1.usd",
        ],
        "jetbot": [
            "/Isaac/Robots/Jetbot/jetbot.usd",
        ],
        "create3": [
            "/Isaac/Robots/iRobot/create_3/create_3.usd",
        ],
        "kaya": [
            "/Isaac/Robots/Kaya/kaya.usd",
        ],
        "spot": [
            "/Isaac/Robots/BostonDynamics/spot/spot.usd",
            "/Isaac/Robots/Spot/spot.usd",
        ],
        "go2": [
            "/Isaac/Robots/Unitree/Go2/go2.usd",
        ],
        "anymal_c": [
            "/Isaac/Robots/ANYbotics/ANYmal_C/anymal_c.usd",
        ],
        "forklift_b": [
            "/Isaac/Robots/Forklift/forklift_b.usd",
            "/Isaac/Props/Forklift/forklift_b.usd",
        ],
        "forklift_c": [
            "/Isaac/Robots/Forklift/forklift_c.usd",
            "/Isaac/Props/Forklift/forklift_c.usd",
        ],
        "dingo": [
            "/Isaac/Robots/Clearpath/Dingo/dingo.usd",
        ],
        "jackal": [
            "/Isaac/Robots/Clearpath/Jackal/jackal.usd",
        ],
        "leatherback": [
            "/Isaac/Robots/Clearpath/Leatherback/leatherback.usd",
        ],
    }
    candidates = robot_assets[robot_key]
    return add_asset_from_candidates(
        prim_path=prim_path,
        relative_candidates=candidates,
        translation=translation,
        rotation_xyz_deg=rotation_xyz_deg,
        scale=scale,
    )


def add_environment(prim_path: str, env_key: str, translation=(0.0, 0.0, 0.0), scale=None):
    env_assets = {
        "warehouse": [
            "/Isaac/Environments/Simple_Warehouse/full_warehouse.usd",
            "/Isaac/Environments/Simple_Warehouse/warehouse.usd",
        ],
        "warehouse_sensors": [
            "/Isaac/Environments/Simple_Warehouse/warehouse_with_forklifts.usd",
            "/Isaac/Environments/Simple_Warehouse/warehouse.usd",
        ],
        "hospital": [
            "/Isaac/Environments/Hospital/hospital.usd",
            "/Isaac/Environments/Office/office.usd",
        ],
        "office": [
            "/Isaac/Environments/Office/office.usd",
            "/Isaac/Environments/Grid/default_environment.usd",
        ],
        "grid": [
            "/Isaac/Environments/Grid/default_environment.usd",
        ],
        "room": [
            "/Isaac/Environments/Simple_Room/simple_room.usd",
            "/Isaac/Environments/Grid/default_environment.usd",
        ],
    }
    candidates = env_assets[env_key]
    return add_asset_from_candidates(
        prim_path=prim_path,
        relative_candidates=candidates,
        translation=translation,
        scale=scale,
    )


def add_box_row(parent_path: str, base_name: str, start, count: int, delta, scale=(0.5, 0.5, 0.5)):
    sx, sy, sz = start
    dx, dy, dz = delta
    for index in range(count):
        add_box(
            prim_path=f"{parent_path}/{base_name}_{index:02d}",
            translation=(sx + index * dx, sy + index * dy, sz + index * dz),
            scale=scale,
        )


def add_box_grid(parent_path: str, base_name: str, start, rows: int, cols: int, spacing, scale=(0.5, 0.5, 0.5)):
    sx, sy, sz = start
    dx, dy = spacing
    for row in range(rows):
        for col in range(cols):
            add_box(
                prim_path=f"{parent_path}/{base_name}_{row:02d}_{col:02d}",
                translation=(sx + row * dx, sy + col * dy, sz),
                scale=scale,
            )


def add_corridor_walls(parent_path: str, length: int, lane_width: float, wall_height: float, x_start: float = -8.0):
    for i in range(length):
        x = x_start + i * 1.5
        add_box(f"{parent_path}/left_wall_{i:02d}", (x, lane_width, wall_height * 0.5), (0.3, 0.3, wall_height))
        add_box(f"{parent_path}/right_wall_{i:02d}", (x, -lane_width, wall_height * 0.5), (0.3, 0.3, wall_height))


def add_maze(parent_path: str, x0: float, y0: float, pattern):
    for row_index, row in enumerate(pattern):
        for col_index, cell in enumerate(row):
            if cell == 1:
                add_box(
                    f"{parent_path}/maze_{row_index:02d}_{col_index:02d}",
                    (x0 + row_index * 1.4, y0 + col_index * 1.4, 0.8),
                    (0.6, 0.6, 1.6),
                )


print("Utility cell loaded successfully.")


## 01. Warehouse Mixed Fleet Complex

**Scene Name:** `WarehouseMixedFleetComplex`


In [ ]:
scene_root = build_scene_root("WarehouseMixedFleetComplex")
add_environment(f"{scene_root}/Environment", "warehouse")
add_robot(f"{scene_root}/NovaCarter", "nova_carter", translation=(-5.0, 0.0, 0.0))
add_robot(f"{scene_root}/ForkliftC", "forklift_c", translation=(2.0, -3.0, 0.0), rotation_xyz_deg=(0.0, 0.0, 90.0))
add_robot(f"{scene_root}/Franka", "franka", translation=(6.5, 2.5, 0.0), rotation_xyz_deg=(0.0, 0.0, -90.0))
add_box_row(f"{scene_root}/PalletStacks", "pallet", start=(-6.0, -5.0, 0.35), count=7, delta=(2.0, 0.0, 0.0), scale=(0.8, 0.8, 0.35))
add_box_row(f"{scene_root}/CargoLaneA", "cargo", start=(-5.5, 4.5, 0.4), count=6, delta=(1.8, 0.0, 0.0), scale=(0.6, 0.6, 0.4))
add_corridor_walls(f"{scene_root}/Corridor", length=10, lane_width=2.2, wall_height=1.6, x_start=-8.0)


## 02. Dual Arm Sorting Cell Complex

**Scene Name:** `DualArmSortingCellComplex`


In [ ]:
scene_root = build_scene_root("DualArmSortingCellComplex")
add_environment(f"{scene_root}/Environment", "room")
add_robot(f"{scene_root}/FrankaLeft", "franka", translation=(-1.6, 1.4, 0.0), rotation_xyz_deg=(0.0, 0.0, -90.0))
add_robot(f"{scene_root}/FrankaRight", "franka", translation=(1.6, 1.4, 0.0), rotation_xyz_deg=(0.0, 0.0, -90.0))
add_robot(f"{scene_root}/Jetbot", "jetbot", translation=(0.0, -3.2, 0.0))
add_box(f"{scene_root}/MainTable", translation=(0.0, 0.0, 0.45), scale=(2.8, 1.3, 0.45))
add_box_row(f"{scene_root}/BinsLeft", "bin", start=(-3.0, -0.5, 0.3), count=4, delta=(0.0, 1.0, 0.0), scale=(0.5, 0.5, 0.3))
add_box_row(f"{scene_root}/BinsRight", "bin", start=(3.0, -0.5, 0.3), count=4, delta=(0.0, 1.0, 0.0), scale=(0.5, 0.5, 0.3))
add_box_row(f"{scene_root}/Conveyor", "segment", start=(-2.4, -1.8, 0.2), count=7, delta=(0.8, 0.0, 0.0), scale=(0.35, 0.9, 0.2))


## 03. Hospital Inspection And Service Complex

**Scene Name:** `HospitalInspectionAndServiceComplex`


In [ ]:
scene_root = build_scene_root("HospitalInspectionAndServiceComplex")
add_environment(f"{scene_root}/Environment", "hospital")
add_robot(f"{scene_root}/NovaCarter", "nova_carter", translation=(-4.0, 0.0, 0.0))
add_robot(f"{scene_root}/Spot", "spot", translation=(2.0, -2.5, 0.0))
add_robot(f"{scene_root}/Franka", "franka", translation=(5.5, 3.0, 0.0), rotation_xyz_deg=(0.0, 0.0, 180.0))
add_box_row(f"{scene_root}/Beds", "bed", start=(-1.5, 4.0, 0.35), count=4, delta=(2.0, 0.0, 0.0), scale=(1.4, 0.8, 0.35))
add_box_row(f"{scene_root}/MedicalSupplies", "supply", start=(-3.5, -4.0, 0.25), count=6, delta=(1.2, 0.0, 0.0), scale=(0.4, 0.4, 0.25))
add_corridor_walls(f"{scene_root}/CorridorWalls", length=8, lane_width=1.6, wall_height=1.8, x_start=-5.0)


## 04. Office Delivery Multi Robot Complex

**Scene Name:** `OfficeDeliveryMultiRobotComplex`


In [ ]:
scene_root = build_scene_root("OfficeDeliveryMultiRobotComplex")
add_environment(f"{scene_root}/Environment", "office")
add_robot(f"{scene_root}/Jetbot", "jetbot", translation=(-5.0, -3.0, 0.0))
add_robot(f"{scene_root}/Create3", "create3", translation=(-1.0, -3.0, 0.0))
add_robot(f"{scene_root}/Kaya", "kaya", translation=(3.0, -3.0, 0.0))
add_robot(f"{scene_root}/Dingo", "dingo", translation=(6.0, -3.0, 0.0))
add_box_grid(f"{scene_root}/OfficeDesks", "desk", start=(-4.0, -1.0, 0.4), rows=3, cols=3, spacing=(3.0, 2.2), scale=(1.1, 0.7, 0.4))
add_box_row(f"{scene_root}/Packages", "package", start=(-3.5, 4.5, 0.25), count=8, delta=(1.3, 0.0, 0.0), scale=(0.35, 0.35, 0.25))


## 05. Quadruped Maze And Ramp Challenge

**Scene Name:** `QuadrupedMazeAndRampChallenge`


In [ ]:
scene_root = build_scene_root("QuadrupedMazeAndRampChallenge")
add_environment(f"{scene_root}/Environment", "grid")
add_robot(f"{scene_root}/Spot", "spot", translation=(-7.0, -5.0, 0.0))
add_robot(f"{scene_root}/Go2", "go2", translation=(-7.0, 0.0, 0.0))
add_robot(f"{scene_root}/ANYmalC", "anymal_c", translation=(-7.0, 5.0, 0.0))
maze_pattern = [
    [1,1,1,0,1,1,0,1],
    [0,0,1,0,0,1,0,1],
    [1,0,1,1,0,1,0,1],
    [1,0,0,1,0,0,0,1],
    [1,1,0,1,1,1,0,1],
    [0,1,0,0,0,1,0,0],
]
add_maze(f"{scene_root}/Maze", x0=-2.0, y0=-5.0, pattern=maze_pattern)
add_ramp(f"{scene_root}/RampA", translation=(7.0, -3.5, 0.8), scale=(3.0, 1.6, 0.25), rotation_xyz_deg=(0.0, -18.0, 0.0))
add_ramp(f"{scene_root}/RampB", translation=(7.0, 3.5, 0.8), scale=(3.0, 1.6, 0.25), rotation_xyz_deg=(0.0, 18.0, 0.0))
add_box(f"{scene_root}/PlatformA", translation=(10.0, -3.5, 1.2), scale=(1.5, 1.5, 0.25))
add_box(f"{scene_root}/PlatformB", translation=(10.0, 3.5, 1.2), scale=(1.5, 1.5, 0.25))


## 06. Warehouse Digital Twin Loading Zone Complex

**Scene Name:** `WarehouseDigitalTwinLoadingZoneComplex`


In [ ]:
scene_root = build_scene_root("WarehouseDigitalTwinLoadingZoneComplex")
add_environment(f"{scene_root}/Environment", "warehouse_sensors")
add_robot(f"{scene_root}/Leatherback", "leatherback", translation=(-5.0, -2.0, 0.0))
add_robot(f"{scene_root}/ForkliftB", "forklift_b", translation=(0.5, -2.5, 0.0), rotation_xyz_deg=(0.0, 0.0, 90.0))
add_robot(f"{scene_root}/Jackal", "jackal", translation=(5.0, -2.0, 0.0))
add_robot(f"{scene_root}/Franka", "franka", translation=(7.0, 3.0, 0.0), rotation_xyz_deg=(0.0, 0.0, 180.0))
add_box_row(f"{scene_root}/InboundPallets", "pallet", start=(-6.0, 4.5, 0.35), count=5, delta=(1.8, 0.0, 0.0), scale=(0.8, 0.8, 0.35))
add_box_row(f"{scene_root}/OutboundBoxes", "box", start=(1.0, 4.5, 0.3), count=6, delta=(1.2, 0.0, 0.0), scale=(0.5, 0.5, 0.3))
add_box_row(f"{scene_root}/TransferConveyor", "segment", start=(-1.5, 1.0, 0.2), count=8, delta=(0.8, 0.0, 0.0), scale=(0.35, 0.9, 0.2))


## 07. Retail Store Service Robots

**Scene Name:** `RetailStoreServiceRobots`


In [ ]:
scene_root = build_scene_root("RetailStoreServiceRobots")
add_environment(f"{scene_root}/Environment", "grid")
add_robot(f"{scene_root}/Jetbot", "jetbot", translation=(-6.0, -4.0, 0.0))
add_robot(f"{scene_root}/Create3", "create3", translation=(-2.0, -4.0, 0.0))
add_robot(f"{scene_root}/Kaya", "kaya", translation=(2.0, -4.0, 0.0))
add_box_row(f"{scene_root}/ShelfRowA", "shelf", start=(-5.0, -1.5, 1.0), count=6, delta=(2.0, 0.0, 0.0), scale=(0.5, 2.5, 1.0))
add_box_row(f"{scene_root}/ShelfRowB", "shelf", start=(-5.0, 2.5, 1.0), count=6, delta=(2.0, 0.0, 0.0), scale=(0.5, 2.5, 1.0))
add_box_grid(f"{scene_root}/CheckoutZones", "checkout", start=(6.0, -2.0, 0.5), rows=1, cols=3, spacing=(0.0, 2.5), scale=(1.0, 0.8, 0.5))


## 08. Factory Assembly Line Inspection

**Scene Name:** `FactoryAssemblyLineInspection`


In [ ]:
scene_root = build_scene_root("FactoryAssemblyLineInspection")
add_environment(f"{scene_root}/Environment", "room")
add_robot(f"{scene_root}/FrankaA", "franka", translation=(-3.0, 1.5, 0.0), rotation_xyz_deg=(0.0, 0.0, -90.0))
add_robot(f"{scene_root}/FrankaB", "franka", translation=(3.0, 1.5, 0.0), rotation_xyz_deg=(0.0, 0.0, -90.0))
add_robot(f"{scene_root}/Jackal", "jackal", translation=(0.0, -4.0, 0.0))
add_box_row(f"{scene_root}/AssemblyLine", "segment", start=(-5.0, -0.8, 0.25), count=10, delta=(1.0, 0.0, 0.0), scale=(0.45, 1.1, 0.25))
add_box_row(f"{scene_root}/InspectionStations", "station", start=(-4.0, 3.6, 0.6), count=4, delta=(2.6, 0.0, 0.0), scale=(0.7, 0.9, 0.6))


## 09. Airport Baggage Handling Zone

**Scene Name:** `AirportBaggageHandlingZone`


In [ ]:
scene_root = build_scene_root("AirportBaggageHandlingZone")
add_environment(f"{scene_root}/Environment", "grid")
add_robot(f"{scene_root}/Dingo", "dingo", translation=(-6.0, -4.0, 0.0))
add_robot(f"{scene_root}/ForkliftC", "forklift_c", translation=(1.0, -4.5, 0.0))
add_box_row(f"{scene_root}/BaggageBeltsA", "belt", start=(-6.0, -1.0, 0.2), count=10, delta=(1.2, 0.0, 0.0), scale=(0.5, 0.8, 0.2))
add_box_row(f"{scene_root}/BaggageBeltsB", "belt", start=(-6.0, 1.5, 0.2), count=10, delta=(1.2, 0.0, 0.0), scale=(0.5, 0.8, 0.2))
add_box_grid(f"{scene_root}/SuitcaseGrid", "suitcase", start=(-5.5, 4.0, 0.3), rows=3, cols=5, spacing=(1.2, 1.2), scale=(0.4, 0.3, 0.3))


## 10. Laboratory Mobile Manipulation

**Scene Name:** `LaboratoryMobileManipulation`


In [ ]:
scene_root = build_scene_root("LaboratoryMobileManipulation")
add_environment(f"{scene_root}/Environment", "room")
add_robot(f"{scene_root}/NovaCarter", "nova_carter", translation=(-4.0, -2.0, 0.0))
add_robot(f"{scene_root}/Franka", "franka", translation=(3.5, 2.5, 0.0), rotation_xyz_deg=(0.0, 0.0, 180.0))
add_box_grid(f"{scene_root}/LabBenches", "bench", start=(-3.0, 0.0, 0.45), rows=2, cols=3, spacing=(3.0, 2.0), scale=(1.2, 0.8, 0.45))
add_box_row(f"{scene_root}/Instruments", "instrument", start=(-4.5, 4.5, 0.35), count=7, delta=(1.2, 0.0, 0.0), scale=(0.35, 0.35, 0.35))


## 11. Smart Home Assistance

**Scene Name:** `SmartHomeAssistance`


In [ ]:
scene_root = build_scene_root("SmartHomeAssistance")
add_environment(f"{scene_root}/Environment", "room")
add_robot(f"{scene_root}/Create3", "create3", translation=(-5.0, -4.0, 0.0))
add_robot(f"{scene_root}/Jetbot", "jetbot", translation=(-1.5, -4.0, 0.0))
add_box(f"{scene_root}/Sofa", translation=(-2.5, 1.5, 0.5), scale=(2.2, 0.9, 0.5))
add_box(f"{scene_root}/DiningTable", translation=(3.0, 1.0, 0.45), scale=(1.4, 1.2, 0.45))
add_box_grid(f"{scene_root}/Cabinets", "cabinet", start=(5.5, -2.5, 0.8), rows=1, cols=3, spacing=(0.0, 1.5), scale=(0.6, 0.5, 0.8))
add_box_row(f"{scene_root}/SmallObjects", "object", start=(-5.0, 4.2, 0.2), count=8, delta=(1.2, 0.0, 0.0), scale=(0.2, 0.2, 0.2))


## 12. Construction Site Material Transport

**Scene Name:** `ConstructionSiteMaterialTransport`


In [ ]:
scene_root = build_scene_root("ConstructionSiteMaterialTransport")
add_environment(f"{scene_root}/Environment", "grid")
add_robot(f"{scene_root}/ForkliftB", "forklift_b", translation=(-4.5, -4.0, 0.0))
add_robot(f"{scene_root}/Jackal", "jackal", translation=(2.5, -4.0, 0.0))
add_box_grid(f"{scene_root}/ConcreteBlocks", "block", start=(-5.0, -0.5, 0.45), rows=3, cols=4, spacing=(2.0, 1.8), scale=(0.7, 0.7, 0.45))
add_box_row(f"{scene_root}/Barriers", "barrier", start=(-6.0, 5.0, 0.6), count=10, delta=(1.2, 0.0, 0.0), scale=(0.4, 0.25, 0.6))
add_ramp(f"{scene_root}/Slope", translation=(7.0, 1.0, 0.7), scale=(3.5, 1.5, 0.25), rotation_xyz_deg=(0.0, -15.0, 0.0))


## 13. Underground Parking Patrol

**Scene Name:** `UndergroundParkingPatrol`


In [ ]:
scene_root = build_scene_root("UndergroundParkingPatrol")
add_environment(f"{scene_root}/Environment", "grid")
add_robot(f"{scene_root}/NovaCarter", "nova_carter", translation=(-6.0, -4.0, 0.0))
add_robot(f"{scene_root}/Spot", "spot", translation=(-2.0, -4.0, 0.0))
add_box_grid(f"{scene_root}/ParkingColumns", "column", start=(-4.0, -1.5, 1.1), rows=3, cols=4, spacing=(3.0, 2.4), scale=(0.35, 0.35, 1.1))
add_box_row(f"{scene_root}/ParkingStops", "car", start=(-5.0, 5.0, 0.55), count=8, delta=(1.8, 0.0, 0.0), scale=(0.8, 1.4, 0.55))
add_corridor_walls(f"{scene_root}/DriveLanes", length=8, lane_width=4.2, wall_height=1.2, x_start=-6.0)


## 14. Disaster Response Indoor Rubble

**Scene Name:** `DisasterResponseIndoorRubble`


In [ ]:
scene_root = build_scene_root("DisasterResponseIndoorRubble")
add_environment(f"{scene_root}/Environment", "grid")
add_robot(f"{scene_root}/Spot", "spot", translation=(-6.0, -4.0, 0.0))
add_robot(f"{scene_root}/Go2", "go2", translation=(-2.5, -4.0, 0.0))
add_robot(f"{scene_root}/Jackal", "jackal", translation=(2.0, -4.0, 0.0))
add_box_grid(f"{scene_root}/RubbleClusters", "rubble", start=(-5.0, -0.5, 0.4), rows=4, cols=5, spacing=(2.0, 1.6), scale=(0.5, 0.5, 0.4))
add_box_row(f"{scene_root}/CollapsedWalls", "wall", start=(-6.0, 5.0, 0.9), count=8, delta=(1.5, 0.0, 0.0), scale=(0.65, 0.25, 0.9))


## 15. Library Delivery And Guidance

**Scene Name:** `LibraryDeliveryAndGuidance`


In [ ]:
scene_root = build_scene_root("LibraryDeliveryAndGuidance")
add_environment(f"{scene_root}/Environment", "office")
add_robot(f"{scene_root}/Create3", "create3", translation=(-5.0, -4.0, 0.0))
add_robot(f"{scene_root}/Jetbot", "jetbot", translation=(-1.0, -4.0, 0.0))
add_box_row(f"{scene_root}/BookShelvesA", "shelf", start=(-4.5, -1.0, 1.0), count=5, delta=(2.2, 0.0, 0.0), scale=(0.45, 2.2, 1.0))
add_box_row(f"{scene_root}/BookShelvesB", "shelf", start=(-4.5, 2.5, 1.0), count=5, delta=(2.2, 0.0, 0.0), scale=(0.45, 2.2, 1.0))
add_box_grid(f"{scene_root}/ReadingDesks", "desk", start=(6.0, -1.5, 0.45), rows=2, cols=2, spacing=(2.2, 2.2), scale=(0.8, 0.8, 0.45))


## 16. Campus Mail Delivery

**Scene Name:** `CampusMailDelivery`


In [ ]:
scene_root = build_scene_root("CampusMailDelivery")
add_environment(f"{scene_root}/Environment", "grid")
add_robot(f"{scene_root}/Dingo", "dingo", translation=(-6.0, -4.0, 0.0))
add_robot(f"{scene_root}/Kaya", "kaya", translation=(-2.5, -4.0, 0.0))
add_box_grid(f"{scene_root}/BuildingBlocks", "building", start=(-4.0, -0.5, 1.3), rows=3, cols=3, spacing=(3.0, 3.0), scale=(1.0, 1.0, 1.3))
add_box_row(f"{scene_root}/MailDropPoints", "mailbox", start=(-5.5, 5.5, 0.7), count=7, delta=(1.8, 0.0, 0.0), scale=(0.35, 0.35, 0.7))


## 17. Supermarket Restocking And Navigation

**Scene Name:** `SupermarketRestockingAndNavigation`


In [ ]:
scene_root = build_scene_root("SupermarketRestockingAndNavigation")
add_environment(f"{scene_root}/Environment", "grid")
add_robot(f"{scene_root}/ForkliftC", "forklift_c", translation=(-6.0, -4.0, 0.0))
add_robot(f"{scene_root}/Jetbot", "jetbot", translation=(-2.0, -4.0, 0.0))
add_robot(f"{scene_root}/Create3", "create3", translation=(2.0, -4.0, 0.0))
add_box_row(f"{scene_root}/AisleShelves1", "shelf", start=(-5.0, -1.5, 1.0), count=6, delta=(2.0, 0.0, 0.0), scale=(0.5, 2.4, 1.0))
add_box_row(f"{scene_root}/AisleShelves2", "shelf", start=(-5.0, 2.5, 1.0), count=6, delta=(2.0, 0.0, 0.0), scale=(0.5, 2.4, 1.0))
add_box_grid(f"{scene_root}/RestockPallets", "pallet", start=(7.0, -1.0, 0.35), rows=1, cols=4, spacing=(0.0, 1.6), scale=(0.7, 0.7, 0.35))


## 18. Data Center Inspection

**Scene Name:** `DataCenterInspection`


In [ ]:
scene_root = build_scene_root("DataCenterInspection")
add_environment(f"{scene_root}/Environment", "room")
add_robot(f"{scene_root}/NovaCarter", "nova_carter", translation=(-5.0, -4.0, 0.0))
add_robot(f"{scene_root}/Spot", "spot", translation=(-1.5, -4.0, 0.0))
add_box_row(f"{scene_root}/ServerRacksA", "rack", start=(-4.5, -1.5, 1.1), count=5, delta=(2.0, 0.0, 0.0), scale=(0.7, 1.0, 1.1))
add_box_row(f"{scene_root}/ServerRacksB", "rack", start=(-4.5, 2.5, 1.1), count=5, delta=(2.0, 0.0, 0.0), scale=(0.7, 1.0, 1.1))
add_corridor_walls(f"{scene_root}/RackCorridor", length=8, lane_width=2.0, wall_height=1.7, x_start=-5.5)


## 19. Cleanroom Precision Transport

**Scene Name:** `CleanroomPrecisionTransport`


In [ ]:
scene_root = build_scene_root("CleanroomPrecisionTransport")
add_environment(f"{scene_root}/Environment", "room")
add_robot(f"{scene_root}/Kaya", "kaya", translation=(-5.0, -4.0, 0.0))
add_robot(f"{scene_root}/Franka", "franka", translation=(5.0, 2.5, 0.0), rotation_xyz_deg=(0.0, 0.0, 180.0))
add_box_grid(f"{scene_root}/CleanStations", "station", start=(-3.5, -0.5, 0.5), rows=2, cols=3, spacing=(3.0, 2.2), scale=(0.9, 0.9, 0.5))
add_box_row(f"{scene_root}/PrecisionTrays", "tray", start=(-4.0, 4.5, 0.2), count=8, delta=(1.1, 0.0, 0.0), scale=(0.25, 0.25, 0.2))


## 20. Restaurant Service And Kitchen Delivery

**Scene Name:** `RestaurantServiceAndKitchenDelivery`


In [ ]:
scene_root = build_scene_root("RestaurantServiceAndKitchenDelivery")
add_environment(f"{scene_root}/Environment", "office")
add_robot(f"{scene_root}/Create3", "create3", translation=(-5.0, -4.0, 0.0))
add_robot(f"{scene_root}/Jetbot", "jetbot", translation=(-1.5, -4.0, 0.0))
add_box_grid(f"{scene_root}/DiningTables", "table", start=(-3.5, -0.5, 0.45), rows=2, cols=3, spacing=(3.0, 2.4), scale=(0.75, 0.75, 0.45))
add_box_row(f"{scene_root}/KitchenCounters", "counter", start=(5.5, -1.5, 0.6), count=3, delta=(0.0, 2.2, 0.0), scale=(1.0, 0.6, 0.6))
add_box_row(f"{scene_root}/FoodPickupArea", "pickup", start=(2.5, 4.5, 0.25), count=5, delta=(1.1, 0.0, 0.0), scale=(0.3, 0.3, 0.25))
